# V3 — Hyperparameter search de Qwen3-1.7B con LoRA

Tres configuraciones de LoRA fine-tuning exploradas para cerrar el gap
respecto a Flan-T5-base V3 y acercarnos al techo BART/Pegasus.

| Config   | epochs | LR   | rank | target_modules      | Hipótesis                            |
|----------|--------|------|------|---------------------|--------------------------------------|
| v3_qwen_A | 1    | 2e-4 | 16   | attn (q,k,v,o)      | Baseline (= V2, sanity check)        |
| v3_qwen_B | 2    | 1e-4 | 16   | attn                | More epochs + lower LR → push R-2    |
| v3_qwen_C | 1    | 2e-4 | 16   | attn + FFN           | Target FFN layers (recent literature) |

Parámetros constantes:
- bf16 + gradient checkpointing
- Batch efectivo = 16 (per_device=1 × grad_accum=16)
- `train_max_input=1024` durante training (1536 en inferencia)
- `lora_alpha=32`, `lora_dropout=0.05`

**Evaluación:** checkpoint recargado limpiamente (ver V2 bug docs) antes
de generación con beam search sobre 200 muestras de test.

**Tiempo estimado:** ~6-8h en total en RTX 5070.

In [ ]:
# Setup
import sys
import os
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

import torch
from src.data.loader import load_config
from src.training.experiments import ExperimentSpec, run_experiment_suite

print(f"CUDA: {torch.cuda.is_available()} | {torch.cuda.get_device_name(0)}")
cfg = load_config("../config/config.yaml")

In [ ]:
# Define the three Qwen3 experiments.
# All share train_max_input=1024 (VRAM constraint) and gradient checkpointing
# (handled inside build_causal_trainer).

qwen_specs = [
    ExperimentSpec(
        name="v3_qwen_A",
        model_key="qwen",
        train_subset=10000,
        num_epochs=1,
        learning_rate=2e-4,
        per_device_batch_size=1,
        gradient_accumulation_steps=16,
        train_max_input=1024,
    ),
    ExperimentSpec(
        name="v3_qwen_B",
        model_key="qwen",
        train_subset=10000,
        num_epochs=2,
        learning_rate=1e-4,
        per_device_batch_size=1,
        gradient_accumulation_steps=16,
        train_max_input=1024,
    ),
    ExperimentSpec(
        name="v3_qwen_C",
        model_key="qwen",
        train_subset=10000,
        num_epochs=1,
        learning_rate=2e-4,
        per_device_batch_size=1,
        gradient_accumulation_steps=16,
        train_max_input=1024,
        lora_overrides={
            "target_modules": [
                "q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj",
            ],
        },
    ),
]

for s in qwen_specs:
    lora_info = f", lora_overrides={s.lora_overrides}" if s.lora_overrides else ""
    print(f"  {s.name}: {s.train_subset} samples, {s.num_epochs} ep, "
          f"lr={s.learning_rate}{lora_info}")

In [ ]:
# Run the full suite. Total expected time: ~6-8h on RTX 5070.
df_qwen = run_experiment_suite(
    cfg=cfg,
    specs=qwen_specs,
    combined_csv_name="v3_qwen_search.csv",
    test_subset_size=200,
)
df_qwen

In [ ]:
# Visualization: ROUGE-L by configuration
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

if len(df_qwen) > 0:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(data=df_qwen, x="experiment", y="rougeL", ax=ax, palette="magma")

    for i, v in enumerate(df_qwen["rougeL"]):
        ax.text(i, v + 0.2, f"{v:.2f}", ha="center", fontsize=10)

    # V2 baseline reference line
    ax.axhline(y=20.53, color="red", linestyle="--", alpha=0.7,
               label="V2 baseline (20.53)")

    ax.set_title("V3 — Qwen3-1.7B LoRA hyperparameter search (ROUGE-L)")
    ax.set_ylabel("ROUGE-L")
    ax.set_xlabel("Experiment")
    ax.set_ylim(bottom=min(df_qwen["rougeL"].min() - 1, 19))
    ax.legend()

    plt.tight_layout()
    plt.savefig("../results/figures/v3_qwen_search.png", bbox_inches="tight", dpi=120)
    plt.show()

In [ ]:
# Identify the winning configuration
if len(df_qwen) > 0:
    best = df_qwen.iloc[0]  # already sorted by rougeL desc
    print(f"\U0001f3c6 Best Qwen3 config: {best['experiment']}")
    print(f"   ROUGE-L: {best['rougeL']:.2f}")
    print(f"   ROUGE-1: {best['rouge1']:.2f}")
    print(f"   ROUGE-2: {best['rouge2']:.2f}")
    print(f"   Delta vs V2 (rougeL=20.53): {best['rougeL'] - 20.53:+.2f}")
    print(f"   Training time: {best['train_minutes']:.1f} min")

## Análisis de V3 — Qwen3-1.7B

*(Rellenar tras ejecución. Puntos a cubrir:)*

1. **Qué configuración ganó y por qué**
2. **Impacto de cada variante:**
   - ¿2 epochs con LR más bajo mejoró ROUGE-2 (hipótesis de V2)?
   - ¿Incluir FFN layers en LoRA aportó mejora significativa?
3. **Comparativa cruzada T5 vs Qwen3** — mejor config de cada familia.
4. **Selección del modelo ganador global** para V3b (LLM-judge + interpretabilidad).